In [0]:
from datetime import datetime
from pyspark.sql import functions as F
 
CONFIG = {
    "catalog":      "workspace",
    "schema":       "banking_datasentry",
    "volume_path":  "/Volumes/workspace/banking_datasentry/datasentry_files",
}
 
TABLE_CONFIG = [
    {"pattern": "customers",    "table": "bronze_customers",     "required": True},
    {"pattern": "accounts",     "table": "bronze_accounts",      "required": True},
    {"pattern": "transactions", "table": "bronze_transactions",  "required": True},
]
 
INGESTED_AT = datetime.now().isoformat()
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Catalog  : {CONFIG['catalog']}")
print(f"Schema   : {CONFIG['schema']}")
print(f"Volume   : {CONFIG['volume_path']}")
print(f"Run time : {INGESTED_AT}")

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze_ingestion_log (
        log_id          STRING,
        file_path       STRING,
        file_name       STRING,
        target_table    STRING,
        source_rows     LONG,
        ingested_rows   LONG,
        reconciliation  STRING,
        status          STRING,
        ingested_at     STRING
    )
    USING DELTA
""")
 
print("bronze_ingestion_log table ready.")

In [0]:
def get_ingested_files() -> set:
    """Return a set of file paths already successfully ingested."""
    log_df = spark.read.format("delta").table("bronze_ingestion_log")
    ingested = log_df.filter(F.col("status") == "SUCCESS") \
                     .select("file_path") \
                     .distinct() \
                     .collect()
    return {row["file_path"] for row in ingested}
 
ingested_files = get_ingested_files()
print(f"Files already ingested: {len(ingested_files)}")
for f in ingested_files:
    print(f"  {f}")

In [0]:
def discover_new_files(volume_path: str, pattern: str, ingested_files: set) -> list:
    """
    List files in volume_path starting with pattern and ending in .csv.
    Excludes files already recorded in the ingestion log.
    """
    try:
        all_files = dbutils.fs.ls(volume_path)
    except Exception as e:
        print(f"  ERROR: Could not list Volume path {volume_path}: {e}")
        return []
 
    matched = [
        f for f in all_files
        if f.name.startswith(pattern)
        and f.name.endswith(".csv")
        and f.path not in ingested_files
    ]
    return matched

In [0]:
import uuid
 
def ingest_with_reconciliation(file_info, table_name: str, ingested_at: str):
    """
    Ingest a single CSV file into a Bronze Delta table with reconciliation.
    Writes result to bronze_ingestion_log.
    """
    file_path = file_info.path
    file_name = file_info.name
 
    print(f"  Reading : {file_path}")
 
    # Step 1 — Count source rows before ingestion (ground truth)
    source_df   = spark.read.option("header", True).option("inferSchema", False).csv(file_path)
    source_rows = source_df.count()
    print(f"  Source rows  : {source_rows:,}")
 
    # Step 2 — Add audit columns
    ingest_df = source_df \
        .withColumn("ingested_at",  F.lit(ingested_at)) \
        .withColumn("source_file",  F.lit(file_name))
 
    # Step 3 — Append to Bronze Delta table
    ingest_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(table_name)
 
    ingested_rows = ingest_df.count()
    print(f"  Ingested rows: {ingested_rows:,}")
 
    # Step 4 — Reconciliation
    reconciliation = "MATCH" if source_rows == ingested_rows else "MISMATCH"
    status         = "SUCCESS" if reconciliation == "MATCH" else "WARNING"
 
    if reconciliation == "MISMATCH":
        print(f"  WARNING: Reconciliation MISMATCH — source={source_rows:,}, ingested={ingested_rows:,}")
    else:
        print(f"  Reconciliation: {reconciliation} ✓")
 
    # Step 5 — Write to ingestion log
    log_entry = spark.createDataFrame([{
        "log_id":         str(uuid.uuid4()),
        "file_path":      file_path,
        "file_name":      file_name,
        "target_table":   table_name,
        "source_rows":    source_rows,
        "ingested_rows":  ingested_rows,
        "reconciliation": reconciliation,
        "status":         status,
        "ingested_at":    ingested_at,
    }])
 
    log_entry.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("bronze_ingestion_log")
 
    return source_rows, ingested_rows, reconciliation

In [0]:
print("=" * 60)
print("BRONZE INGESTION — START")
print("=" * 60)
 
# Reload ingested files at start of run
ingested_files = get_ingested_files()
 
ingestion_summary = []
 
for config in TABLE_CONFIG:
    pattern  = config["pattern"]
    table    = config["table"]
    required = config["required"]
 
    print(f"\n[{pattern.upper()}]")
 
    # Discover new files only
    new_files = discover_new_files(CONFIG["volume_path"], pattern, ingested_files)
 
    if not new_files:
        print(f"  No new files found — already ingested or not present. Skipping.")
        ingestion_summary.append({
            "table": table, "files_found": 0,
            "rows_ingested": 0, "reconciliation": "N/A", "status": "SKIPPED"
        })
        continue
 
    print(f"  New files found: {len(new_files)}")
 
    total_source   = 0
    total_ingested = 0
    overall_recon  = "MATCH"
 
    for file_info in new_files:
        try:
            src, ing, recon = ingest_with_reconciliation(file_info, table, INGESTED_AT)
            total_source   += src
            total_ingested += ing
            if recon == "MISMATCH":
                overall_recon = "MISMATCH"
        except Exception as e:
            print(f"  ERROR ingesting {file_info.path}: {e}")
            overall_recon = "ERROR"
 
    ingestion_summary.append({
        "table":         table,
        "files_found":   len(new_files),
        "rows_ingested": total_ingested,
        "reconciliation": overall_recon,
        "status":        "SUCCESS" if overall_recon == "MATCH" else "WARNING"
    })
 
print("\n" + "=" * 60)
print("BRONZE INGESTION — COMPLETE")
print("=" * 60)

In [0]:
#Ingestion Summary
print(f"\n{'TABLE':<30} {'FILES':>6} {'ROWS':>12} {'RECON':<12} {'STATUS':<10}")
print("-" * 74)
for row in ingestion_summary:
    print(f"{row['table']:<30} {row['files_found']:>6} {row['rows_ingested']:>12,} {row['reconciliation']:<12} {row['status']:<10}")

In [0]:
#View Ingestion Log
print("\nIngestion Log (all runs):")
spark.read.format("delta").table("bronze_ingestion_log") \
    .select("file_name", "target_table", "source_rows",
            "ingested_rows", "reconciliation", "status", "ingested_at") \
    .orderBy("ingested_at") \
    .show(truncate=50)

In [0]:
#Verifying Bronze Table Row Counts
print("=" * 60)
print("BRONZE TABLE ROW COUNTS")
print("=" * 60)
 
for config in TABLE_CONFIG:
    table = config["table"]
    try:
        count = spark.read.format("delta").table(table).count()
        print(f"  {table:<35} : {count:,} rows")
    except Exception as e:
        print(f"  {table:<35} : ERROR — {e}")
 
print("\nNotebook 02 complete. Bronze Delta tables ready for Silver validation.")